# SKN36 | 임베딩 직전 통합 Splitting (V3 개선판)

**핵심:** 기존 중앙부처·지자체·시설 통합을 유지하면서 PDF의 Markdown 표를 행·열·연도 이름이 함께 검색되도록 재구성합니다. 코드와 노트북은 동일한 `SKN36_auto_splitting.py`를 사용합니다.

**주의:** 이 노트북은 임베딩 벡터를 생성하거나 Qdrant·Neo4j에 접속하지 않습니다. 공식 정책의 자격/지역/금액 승인도 자동 수행하지 않습니다.

필수 입력 4개는 `splitting_inputs/`에 놓습니다. 노령층 분권 PDF 및 기존 4개 PDF 원본은 있을 경우 추가 검수·추출에 사용합니다.

## 1. 현재 준비된 데이터 (업로드 ZIP 실측)

| 구분 | 파일명 | 레코드 | 주의 |
|---|---|---:|---|
| 중앙 정책 | `policies.jsonl` | 461 | 상세 461건 |
| 지자체 정책 | `welfare_all_full_20261107_103452.jsonl` | 1,485 | 경기 485건 상세 미확보 |
| 시설 | `all_regions_elderly_welfare.json` | 164 | 전국 완전 목록 아님 |
| PDF 4종 | `guideline_sections_filtered.jsonl` | 1,189 섹션 | 표/페이지 검수 필요 |
| 노령층 분권 PDF | `2026 나에게 힘이 되는 복지서비스_분권(노령층).pdf` | 별도 원본 1종 | 현재 ZIP에는 없음. 넣으면 임시 페이지 추출 |

필수 데이터는 3개 GitHub 브랜치의 4개 파일로 구성됩니다. 5번째 PDF 원본이나 4개 PDF 원본을 추가하면 분석 범위가 확장되지만, 사람의 원본 대조가 필요합니다.

## 2. 설정 — 필요한 경우 `INPUT_DIR` 한 곳만 수정

노트북과 Python 파일을 같은 상위 폴더에 놓고 Jupyter를 그 폴더에서 실행하면 경로 변경 없이 사용할 수 있습니다.

In [ ]:
from pathlib import Path

# 다운로드한 데이터 4개가 있는 폴더(기본값: 노트북과 같은 폴더/splitting_inputs)
INPUT_DIR = str(Path.cwd() / "splitting_inputs")

# 출력은 구버전 splitting_output과 분리하여 보존
OUTPUT_DIR = str(Path.cwd() / "splitting_output_v3")

REQUIRE_FIFTH_PDF = False    # True 설정 시 노령층 분권이 없으면 중단
STRICT_COUNTS = True        # 현재 기준 461/1485/164/1189 검사
HF_TOKENIZER_NAME = ""     # 임베딩 모델 선정 전에는 비워둘 것
MAX_EMBEDDING_TOKENS = 512
print("입력:", INPUT_DIR)
print("출력:", OUTPUT_DIR)

## 3. 분할 및 데이터 무결성 검사 실행

기존 규칙: 정책 **650자**, PDF 본문 **700자**, 겹침 **90자**. 개선 규칙: Markdown 표는 셀·연도·행 제목을 반복하여 **표 문맥이 사라지는 분할을 줄임**. 표 원본 검수는 여전히 필요합니다.

In [ ]:
import importlib.util, sys
from pathlib import Path

MODULE_FILE = Path.cwd() / "SKN36_auto_splitting.py"
if not MODULE_FILE.is_file():
    raise FileNotFoundError(f"같은 폴더에 Python 파일을 배치하세요: {MODULE_FILE}")
spec = importlib.util.spec_from_file_location("skn36_auto_splitting", MODULE_FILE)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)
manifest = module.run_pipeline(module.Config(
    input_dir=Path(INPUT_DIR), output_dir=Path(OUTPUT_DIR),
    require_fifth_pdf=REQUIRE_FIFTH_PDF,strict_counts=STRICT_COUNTS,
    tokenizer_name=HF_TOKENIZER_NAME,max_embedding_tokens=MAX_EMBEDDING_TOKENS,
))
print("✅ 실행 완료 / 총 청크", manifest["generated"]["total_chunks"])

## 4. 자동 생성된 품질 보고서 확인

데이터가 준비됐다는 뜻과 정책·금액·페이지가 공식 검증됐다는 뜻은 다릅니다. 아래 `limitations`를 반드시 팀원과 공유하세요.

In [ ]:
import json
from pathlib import Path
out=Path(OUTPUT_DIR)
quality=json.loads((out/"split_quality_report.json").read_text(encoding="utf-8"))
for k in ["source_counts","total_chunk_count","embedding_candidate_count", "pdf_table_chunk_count",
          "pdf_table_rows_referenced","pdf_multi_page_chunk_count", "policy_pdf_link_candidates",
          "pdf_sections_with_review_flags","tokenizer_status","page_alignment","limitations",
          "pre_embedding_readiness"]:
    print(k,":",quality.get(k))
assert quality["source_counts"]["total_policies"] == 1946
assert quality["source_counts"]["local_detail_missing"] == 485
print("✅ 자료 건수와 품질 보고서 확인 완료")

## 5. 표의 연도·금액 문맥 보존 확인 (기초연금 사례)

기존 청크는 `<br>100% 349,700원`처럼 시작했지만, 개선판은 **표 제목 / 행 / 2025년(현행) / 2026년(개정안)** 을 함께 제공합니다. 아래 셀은 실제 파일에서 확인합니다.

In [ ]:
import json
with (Path(OUTPUT_DIR)/"guideline_chunks.jsonl").open(encoding="utf-8") as f:
    examples=[json.loads(line) for line in f if '"table_row_sha256"' in line and 'basic_pension_2026_p0013_01' in line]
for c in examples:
    if "기준연금액" in c["body_text"] and "2025년" in c["body_text"] and "2026년" in c["body_text"]:
        print(c["chunk_id"], "\n",c["body_text"][:850])
        break
else:
    print("연도별 비교표 예시 미발견: 입력 파일 구조가 바뀌었는지 검토하세요.")

## 6. Qdrant ↔ Neo4j 참조 무결성 / 저장 대상 검사

In [ ]:
import json
from pathlib import Path

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]
out=Path(OUTPUT_DIR)
emb=read_jsonl(out/"embedding_ready.jsonl")
nodes=read_jsonl(out/"neo4j_nodes.jsonl")
rels=read_jsonl(out/"neo4j_relationships.jsonl")
chunk_ids={n["id"] for n in nodes if n["label"]=="Chunk"}
node_keys={(n["label"],n["id"]) for n in nodes}
assert len(chunk_ids)==sum(n["label"]=="Chunk" for n in nodes)
assert all(p["id"] in chunk_ids for p in emb)
assert all((e["from_label"],e["from_id"]) in node_keys and
           (e["to_label"],e["to_id"]) in node_keys for e in rels)
print(f"✅ 임베딩 후보 {len(emb):,}건, Neo4j 노드 {len(nodes):,}건, 관계 {len(rels):,}건 참조 정상")

## 7. 수동 검수 파일 (자동 생성)

- `policy_region_review.csv`: **1,946건**의 지역 적용 범위 검토. `sido_hint`는 검색 확정 필터가 아닙니다.
- `local_detail_missing_review.csv`: 경기 상세 미확보 **485건** 목록. API 재수집 권한이 필요합니다.
- `pdf_financial_review.csv`: 표·기준금액·연도 등을 공식 PDF와 대조할 목록.
- `citation_page_review.csv`: 여러 페이지 범위인 PDF 청크와 원본 대조 대기 항목.
- `policy_pdf_link_candidates.jsonl`: 문자열 매칭 기반 **검수 후보**, Neo4j에 승인 관계 자동 생성하지 않습니다.
- `cross_pdf_duplicate_candidates.jsonl`: 5번째 PDF가 들어올 때 종합본과 정확 중복 가능성이 있는 부분을 검토하기 위한 큐.

**토크나이저 검증:** `HF_TOKENIZER_NAME`에 실제 임베딩 모델의 토크나이저를 입력하면 청크별 토큰 수를 전수검사합니다. 모델을 결정하지 않았다면 `not_checked_model_not_selected`가 정확한 결과입니다.

**임베딩 이전 완료 기준:** 5번째 PDF 반입 및 수동 검수, 지역·수급 요건 승인, 페이지 인용 검수, 실제 모델 토큰 적합성 검사가 필요합니다. 코드 실행이 성공했다고 이 승인 조건을 충족한 것은 아닙니다.

## 8. PowerShell CLI로 다시 실행

```powershell
cd "C:\Dev\project\project 3st\SKN36-RAG-Splitting"
uv run --no-project python .\SKN36_auto_splitting.py --input-dir ".\splitting_inputs" --output-dir ".\splitting_output_v3" --strict-counts
```

현재 원본이 누락된 5번째 PDF를 필수로 요구하려면 `--require-fifth-pdf`를 추가합니다. 이미 존재하는 `splitting_output/`은 덮어쓰지 않습니다.